# Register MCP Services

Registers (or updates) the 5 Hi Genie Orchestrator MCP Services using a **single
HTTP connection** (`base_path: /mcp`) that exposes all 40 tools. Each persona
service uses `include_tool_selectors` to expose only its subset:

- **Architect** — 14 tools: projects, plans, skills, decisions
- **Reviewer** — 14 tools: reviews, quality gates, broad read
- **Worker** — 17 tools: records, sessions, execution
- **Dispatcher** — 16 tools: orchestration, lifecycle
- **Orchestrator** — all 39 tools (empty selector = all)

**Architecture:** One connection → many services. Tool scoping is YAML-driven
(the `PERSONA_TOOL_SELECTORS` dict below), not app-level routing. Credential
rotation touches one connection. UC enforces tool boundaries — `tools/call` for
an unselected tool returns `-32003`.

**Parameters** (passed via job `base_parameters`):
| Parameter | Description |
|-----------|-------------|
| `catalog` | UC catalog where MCP Services are registered |
| `schema` | UC schema where MCP Services are registered |
| `mcp_connection_name` | Single HTTP connection (base_path: /mcp, all tools) |
| `orchestrator_mcp_service_name` | Full orchestrator MCP Service name |
| `architect_mcp_service_name` | Architect MCP Service name |
| `reviewer_mcp_service_name` | Reviewer MCP Service name |
| `worker_mcp_service_name` | Worker MCP Service name |
| `dispatcher_mcp_service_name` | Dispatcher MCP Service name |
| `workers_group_name` | Group receiving EXECUTE grant on all services |
| `mcp_service_rate_limit_qpm` | Service-wide rate limit (queries/min) |
| `mcp_service_user_rate_limit_qpm` | Per-user rate limit (queries/min) |

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound, ResourceDoesNotExist

# Parameters from job base_parameters (widgets)
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
workers_group_name = dbutils.widgets.get("workers_group_name")
mcp_connection_name = dbutils.widgets.get("mcp_connection_name")

w = WorkspaceClient()

# ---------------------------------------------------------------------------
# YAML-driven tool selectors per persona.
# Each key maps to include_tool_selectors for that service.
# Supports exact match ("tool_name") and prefix match ("prefix_*").
# Empty list [] = expose ALL tools (used for the full orchestrator).
#
# To externalize: load from a YAML fixture file instead of this dict.
# ---------------------------------------------------------------------------
PERSONA_TOOL_SELECTORS: dict[str, list[str]] = {
    "architect": [
        # Projects, plans, skills, design decisions (14 tools)
        # Source: server/mcp/index.ts ARCHITECT_TOOLS
        "create_project",
        "create_plan",
        "get_plan",
        "publish_plan",
        "get_skill_content",
        "update_skill",
        "record_decision",
        "list_projects",
        "get_project_context",
        "get_workstream_context",
        "search_decisions",
        "get_recent_session_summaries",
        "get_health",
        "get_schema_version",
    ],
    "reviewer": [
        # Reviews, quality gates, broad read (14 tools)
        # Source: server/mcp/index.ts REVIEWER_TOOLS
        "create_review",
        "get_review",
        "deliver_remediation",
        "approve_review",
        "list_projects",
        "get_project_context",
        "get_workstream_context",
        "search_decisions",
        "get_recent_session_summaries",
        "get_relevant_errors",
        "get_task_history",
        "get_execution_summary",
        "get_health",
        "get_schema_version",
    ],
    "worker": [
        # Records, sessions, task execution (17 tools)
        # Source: server/mcp/index.ts WORKER_TOOLS
        "record_decision",
        "record_session_summary",
        "record_error",
        "record_workstream_event",
        "record_task_attempt",
        "record_task_completion",
        "start_session",
        "end_session",
        "update_recovery_state",
        "search_context",
        "get_session_context",
        "get_skill_content",
        "get_project_context",
        "get_workstream_context",
        "get_relevant_errors",
        "get_task_history",
        "get_schema_version",
    ],
    "dispatcher": [
        # Orchestration, workstream lifecycle (16 tools)
        # Source: server/mcp/index.ts DISPATCHER_TOOLS
        "orchestrate_plan",
        "materialize_workstreams",
        "list_workstreams",
        "dispatch_workstream",
        "retry_workstream",
        "pause_workstream",
        "unpause_workstream",
        "update_workstream_status",
        "update_task_status",
        "list_projects",
        "get_project_context",
        "get_workstream_context",
        "get_task_history",
        "get_execution_summary",
        "get_health",
        "get_schema_version",
    ],
    "orchestrator": [],  # All tools — no filtering
}

# Service definitions: (widget_key_for_name, persona_key, comment)
SERVICES = [
    (
        dbutils.widgets.get("architect_mcp_service_name"),
        "architect",
        "Hi Genie Orchestrator — architect tools (projects, plans, skills, decisions)",
    ),
    (
        dbutils.widgets.get("reviewer_mcp_service_name"),
        "reviewer",
        "Hi Genie Orchestrator — reviewer tools (reviews, quality gates, broad read)",
    ),
    (
        dbutils.widgets.get("worker_mcp_service_name"),
        "worker",
        "Hi Genie Orchestrator — worker tools (records, sessions, execution)",
    ),
    (
        dbutils.widgets.get("dispatcher_mcp_service_name"),
        "dispatcher",
        "Hi Genie Orchestrator — dispatcher tools (orchestration, lifecycle)",
    ),
    (
        dbutils.widgets.get("orchestrator_mcp_service_name"),
        "orchestrator",
        "Hi Genie Orchestrator — full MCP service (all personas combined)",
    ),
]


def register_or_update_service(
    service_name: str,
    tool_selectors: list[str],
    comment: str,
) -> str:
    """Register or update a single MCP Service.

    All services share one connection (mcp_connection_name) and differ only
    by their include_tool_selectors.
    """
    full_name = f"{catalog}.{schema}.{service_name}"
    expected_conn = f"connections/{mcp_connection_name}"

    try:
        existing = w.api_client.do(
            "GET",
            f"/api/2.1/unity-catalog/mcp-services/{full_name}"
        )
        print(f"  Exists: {full_name}")
        source_conn = existing["config"]["source_connection"]
        is_deleted = source_conn.get("is_deleted", False)
        conn_name_mismatch = source_conn.get("name") != expected_conn
        current_selectors = existing["config"].get("include_tool_selectors", [])
        selectors_changed = sorted(current_selectors) != sorted(tool_selectors)

        if is_deleted or conn_name_mismatch or selectors_changed:
            reasons = []
            if is_deleted:
                reasons.append("connection deleted")
            if conn_name_mismatch:
                reasons.append(f"connection mismatch (have: {source_conn.get('name')})")
            if selectors_changed:
                reasons.append(f"tool selectors changed ({len(current_selectors)}→{len(tool_selectors)})")
            print(f"  ⚠️  Updating: {', '.join(reasons)}")
            w.api_client.do(
                "PATCH",
                f"/api/2.1/unity-catalog/mcp-services/{full_name}",
                query={"update_mask": "config"},
                body={
                    "config": {
                        "source_connection": {"name": expected_conn},
                        "include_tool_selectors": tool_selectors,
                    }
                },
            )
            print(f"  ✅ Updated with {len(tool_selectors) or 'ALL'} tool selectors")
        else:
            print(f"  ✅ Healthy — connection + selectors match")

    except (NotFound, ResourceDoesNotExist):
        print(f"  Creating: {full_name}")
        resp = w.api_client.do(
            "POST",
            "/api/2.1/unity-catalog/mcp-services",
            query={
                "parent": f"schemas/{catalog}.{schema}",
                "mcp_service_id": service_name,
            },
            body={
                "comment": comment,
                "config": {
                    "source_connection": {"name": expected_conn},
                    "include_tool_selectors": tool_selectors,
                },
            },
        )
        print(f"  ✅ Created: {resp['name']} (ID: {resp['id']})")

    return full_name


# Register all services (single connection, per-persona tool selectors)
registered = []
for svc_name, persona_key, comment in SERVICES:
    selectors = PERSONA_TOOL_SELECTORS[persona_key]
    print(f"\n{'='*60}")
    print(f"Registering: {svc_name} (persona={persona_key}, selectors={len(selectors) or 'ALL'})")
    full = register_or_update_service(svc_name, selectors, comment)
    registered.append(full)

print(f"\n{'='*60}")
print(f"✅ All {len(registered)} MCP Services registered (single connection: {mcp_connection_name}).")


In [0]:
import time

# ---------------------------------------------------------------------------
# UC MCP Services discover tools asynchronously after creation/update.
# There is no explicit "initialize" or "refresh" API endpoint — discovery
# is triggered internally when the service config changes.
#
# This cell polls GET .../tools until each service reports its expected
# tool count, with exponential backoff. This replaces the previous
# fire-and-hope pattern that raced against async discovery.
# ---------------------------------------------------------------------------

MAX_WAIT_SECONDS = 120  # Total max wait per service
INITIAL_DELAY = 2.0
BACKOFF_FACTOR = 1.5
MAX_DELAY = 15.0


def wait_for_tools(full_name: str, expected_count: int | None = None) -> list:
    """Poll until tool discovery completes for a service.

    Args:
        full_name: Fully qualified MCP service name (catalog.schema.service)
        expected_count: If set, wait until at least this many tools appear.
                        If None (orchestrator), wait until any tools appear.
    Returns:
        List of discovered tool dicts.
    Raises:
        TimeoutError if tools don't appear within MAX_WAIT_SECONDS.
    """
    delay = INITIAL_DELAY
    elapsed = 0.0

    while elapsed < MAX_WAIT_SECONDS:
        try:
            resp = w.api_client.do(
                "GET",
                f"/api/2.1/unity-catalog/mcp-services/{full_name}/tools"
            )
            tools = resp.get("tools", [])
            # Success conditions:
            # - For persona services: tools count matches expected selectors
            # - For orchestrator (expected_count=None): any tools discovered
            if expected_count is not None and len(tools) >= expected_count:
                return tools
            elif expected_count is None and len(tools) > 0:
                return tools
        except Exception:
            pass  # Service may not be queryable yet

        time.sleep(delay)
        elapsed += delay
        delay = min(delay * BACKOFF_FACTOR, MAX_DELAY)

    raise TimeoutError(
        f"Tool discovery timed out for {full_name} after {MAX_WAIT_SECONDS}s "
        f"(expected={expected_count or 'any'})"
    )


# Wait for all services to complete tool discovery
print("Waiting for tool discovery to complete...\n")
for i, (svc_name, persona_key, _comment) in enumerate(SERVICES):
    full_name = registered[i]
    selectors = PERSONA_TOOL_SELECTORS[persona_key]
    # For persona services, expect exactly len(selectors) tools.
    # For orchestrator (empty selectors = all), expect at least 1.
    expected = len(selectors) if selectors else None

    try:
        tools = wait_for_tools(full_name, expected_count=expected)
        print(f"\u2705 {full_name}: {len(tools)} tools discovered (expected={expected or 'all'})")
    except TimeoutError as e:
        print(f"\u26a0\ufe0f  {e}")
        print(f"   Discovery may still be in progress. Continuing...")

print(f"\n{'='*60}")
print("Tool discovery phase complete.")

In [0]:
# EXECUTE grant allows workers to invoke the MCP Service.
# READ_METADATA allows workers to discover the service and list its tools.
for full_name in registered:
    w.api_client.do(
        "PATCH",
        f"/api/2.1/unity-catalog/permissions/MCP_SERVICE/{full_name}",
        body={
            "changes": [{
                "principal": workers_group_name,
                "add": ["EXECUTE", "READ_METADATA"]
            }]
        }
    )
    print(f"✅ Granted EXECUTE + READ_METADATA on {full_name} to `{workers_group_name}`")

In [0]:
# Rate limits protect the app from being overwhelmed and prevent
# any single agent from monopolizing the service.
service_rate_limit = int(dbutils.widgets.get("mcp_service_rate_limit_qpm"))
user_rate_limit = int(dbutils.widgets.get("mcp_service_user_rate_limit_qpm"))

for full_name in registered:
    try:
        w.api_client.do(
            "PUT",
            f"/api/2.1/unity-catalog/mcp-services/{full_name}/rate-limits",
            body={
                "rate_limits": [
                    {"calls": service_rate_limit, "renewal_period": "MINUTE", "key": "SERVICE"},
                    {"calls": user_rate_limit, "renewal_period": "MINUTE", "key": "USER"},
                ]
            }
        )
        print(f"✅ Rate limits on {full_name}: {service_rate_limit}/{user_rate_limit} QPM")
    except Exception as e:
        print(f"⚠️  {full_name}: rate limits skipped — {e}")

In [0]:
# Final verification — confirm tools, permissions, and rate limits for each service
print(f"{'='*60}")
print("FINAL VERIFICATION SUMMARY")
print(f"{'='*60}")
print(f"\nConnection: {mcp_connection_name}")
print(f"Schema:     {catalog}.{schema}")
print(f"Workers:    {workers_group_name}\n")

for i, (svc_name, persona_key, _comment) in enumerate(SERVICES):
    full_name = registered[i]
    selectors = PERSONA_TOOL_SELECTORS[persona_key]
    print(f"\n── {persona_key.upper()} ({full_name}) ──")
    try:
        # Tools
        tools_resp = w.api_client.do(
            "GET",
            f"/api/2.1/unity-catalog/mcp-services/{full_name}/tools"
        )
        tools = tools_resp.get("tools", [])
        tool_names = [t["name"] for t in tools]
        print(f"  Tools:       {len(tools)} discovered")
        if selectors:
            missing = set(selectors) - set(tool_names)
            if missing:
                print(f"  ⚠️  Missing:   {sorted(missing)}")

        # Service config
        svc_resp = w.api_client.do(
            "GET",
            f"/api/2.1/unity-catalog/mcp-services/{full_name}"
        )
        conn = svc_resp["config"]["source_connection"]["name"]
        stored_selectors = svc_resp["config"].get("include_tool_selectors", [])
        print(f"  Connection:  {conn}")
        print(f"  Selectors:   {len(stored_selectors)} configured")

    except Exception as e:
        print(f"  ⚠️  Error: {e}")

print(f"\n{'='*60}")
print("✅ Registration complete.")